# 03 - Логистическая регрессия

Это наш baseline. Логистическая регрессия устроена просто: умножает каждый признак на свой вес, складывает, прогоняет через сигмоиду и получает вероятность класса 1. Никаких нелинейностей.

Без такой точки отсчёта мне непонятно, что вообще считать хорошим результатом у CatBoost или TabM. Если сложные модели не обгоняют LogReg - значит сигнал в данных в основном линейный, и нейросеть тут особо ничего не даёт. Если обгоняют - тогда архитектура реально что-то ловит.

Сам ноутбук короткий: гружу `train/val/test` из этапа 02, подбираю силу регуляризации `C` на val, на val же подбираю порог отсечения по F2, фиксирую его и считаю метрики на test. В конце сохраняю вероятности и метрики - их потом подхватит ноутбук 08, где собирается общая таблица по всем моделям.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sys.path.append("../src")
import utils

SEED = 42
PROCESSED_DIR = Path("../data/processed")
RESULTS_METRICS = Path("../results/metrics")
RESULTS_PREDS = Path("../results/predictions")
RESULTS_METRICS.mkdir(parents=True, exist_ok=True)
RESULTS_PREDS.mkdir(parents=True, exist_ok=True)

## 1. Загрузка данных

Тут просто читаем три parquet-файла из `data/processed/` (они уже подготовлены в ноутбуке 02) и `feature_types.json` со списками числовых и категориальных признаков.

Важных проверок две. Первая - размеры сплитов 60/20/20: примерно 42k, 14k и 14k строк. Вторая - доля позитивного класса должна быть одинаковой во всех трёх выборках. У нас везде 9%, это работа стратификации из ноутбука 02. Без неё в test случайно могло бы попасть, скажем, 7% или 11%, и сравнивать метрики было бы некорректно.

9% - сильный дисбаланс. Из-за него дальше включаем `class_weight="balanced"` и оптимизируем F2, а не accuracy.

In [2]:
train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val   = pd.read_parquet(PROCESSED_DIR / "val.parquet")
test  = pd.read_parquet(PROCESSED_DIR / "test.parquet")

with open(PROCESSED_DIR / "feature_types.json") as f:
    feat = json.load(f)

NUM_FEATS = feat["numeric"]
CAT_FEATS = feat["categorical"]

print(f"train: {train.shape}  val: {val.shape}  test: {test.shape}")
print(f"Доля позитивного класса - train: {train['target'].mean():.3f}, val: {val['target'].mean():.3f}, test: {test['target'].mean():.3f}")

train: (41982, 26)  val: (13994, 26)  test: (13994, 26)
Доля позитивного класса - train: 0.090, val: 0.090, test: 0.090


## 2. Трансформер признаков

LogReg не понимает ни строк, ни признаков с разными масштабами. Поэтому всё надо перевести в числа и желательно стандартизировать.

Числовые колонки прогоняю через `StandardScaler` - у каждого признака вычитаю среднее и делю на стандартное отклонение. Иначе L2-регуляризация будет несправедливо штрафовать признаки, у которых единицы измерения покрупнее. К реальной важности признака это никак не относится, но веса бы получились перекошенные.

Категориальные кодирую `OneHotEncoder`: каждое значение становится отдельной бинарной колонкой. `handle_unknown="ignore"` нужен на случай, если в val или test всплывёт категория, которой не было в train - кодировщик не упадёт, просто выдаст нули. `sparse_output=False` - чтобы получить обычный numpy-массив, разреженная матрица тут только мешала бы.

Собирает всё это `ColumnTransformer`: применяет нужное преобразование к нужным колонкам и склеивает результат. Препроцессор обучается только на train, на val и test используется уже как есть. Это автоматически обеспечивает `Pipeline` в следующих ячейках - утечки статистик из val/test в препроцессор не происходит.

In [3]:
X_train, y_train = train[NUM_FEATS + CAT_FEATS], train["target"].to_numpy()
X_val,   y_val   = val  [NUM_FEATS + CAT_FEATS], val  ["target"].to_numpy()
X_test,  y_test  = test [NUM_FEATS + CAT_FEATS], test ["target"].to_numpy()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATS),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATS),
    ]
)

print(f"Числовых признаков: {len(NUM_FEATS)}, категориальных: {len(CAT_FEATS)}")

Числовых признаков: 12, категориальных: 13


## 3. Подбор гиперпараметра C на валидации

`C` в sklearn - это обратная сила L2-регуляризации. Маленькое значение означает сильную регуляризацию (модель не даёт весам уходить далеко от нуля, рискует недоучиться). Большое - наоборот, веса свободные, и модель легко переобучается. Оптимум где-то посередине.

Перебираю 13 значений от 0.001 до 100 по логарифмической шкале - эффект регуляризации меняется по порядкам величины, а не линейно. Для каждого `C` обучаю Pipeline (препроцессор + LogReg) на train, считаю вероятности на val, ищу порог по F2 на val, записываю метрики. В конце беру `C` с лучшим F2.


Если посмотреть на таблицу, F2 у меня болтается в пределах 0.358-0.361. То есть LogReg на этих данных к `C` почти равнодушна. Формально лучшим оказалось `C ≈ 0.12`, но честно - почти любое другое значение из сетки дало бы то же самое.

In [4]:
C_grid = np.logspace(-3, 2, 13)
results_cv = []

for C in C_grid:
    pipe = Pipeline([
        ("prep", preprocessor),
        ("clf",  LogisticRegression(
            C=C, class_weight="balanced",
            max_iter=1000, random_state=SEED, solver="lbfgs"
        )),
    ])
    pipe.fit(X_train, y_train)
    proba_val = pipe.predict_proba(X_val)[:, 1]
    thr = utils.find_best_threshold_f2(y_val, proba_val)
    m = utils.compute_metrics(y_val, proba_val, thr)
    results_cv.append({"C": C, "threshold": thr, **m})
    print(f"C={C:.2f}  thr={thr:.3f}  roc_auc={m['roc_auc']:.4f}  recall={m['recall']:.4f}  f2={m['f2']:.4f}")

cv_df = pd.DataFrame(results_cv).set_index("C")
best_C = float(cv_df["f2"].idxmax())
print(f"\nЛучший C = {best_C}  (F2 на val = {cv_df.loc[best_C, 'f2']:.4f})")
cv_df

C=0.00  thr=0.455  roc_auc=0.6353  recall=0.7038  f2=0.3584
C=0.00  thr=0.426  roc_auc=0.6409  recall=0.7747  f2=0.3597
C=0.01  thr=0.414  roc_auc=0.6438  recall=0.7962  f2=0.3597
C=0.02  thr=0.414  roc_auc=0.6452  recall=0.7874  f2=0.3604
C=0.05  thr=0.417  roc_auc=0.6457  recall=0.7707  f2=0.3601
C=0.12  thr=0.424  roc_auc=0.6462  recall=0.7492  f2=0.3608
C=0.32  thr=0.397  roc_auc=0.6461  recall=0.8161  f2=0.3596
C=0.83  thr=0.396  roc_auc=0.6457  recall=0.8185  f2=0.3592
C=2.15  thr=0.395  roc_auc=0.6453  recall=0.8209  f2=0.3585
C=5.62  thr=0.395  roc_auc=0.6453  recall=0.8225  f2=0.3603
C=14.68  thr=0.395  roc_auc=0.6447  recall=0.8217  f2=0.3593
C=38.31  thr=0.395  roc_auc=0.6453  recall=0.8225  f2=0.3593
C=100.00  thr=0.396  roc_auc=0.6452  recall=0.8185  f2=0.3592

Лучший C = 0.12115276586285889  (F2 на val = 0.3608)


,threshold,roc_auc,precision,recall,f2,tp,fp,fn,tn
C,,,,,,,,,
0.001000,0.455400,0.635319,0.120930,0.703822,0.358359,884,6426,372,6312
0.002610,0.426099,0.640859,0.114444,0.774682,0.359678,973,7529,283,5209
0.006813,0.413828,0.643826,0.112663,0.796178,0.359712,1000,7876,256,4862
0.017783,0.413732,0.645162,0.113730,0.787420,0.360423,989,7707,267,5031
0.046416,0.416982,0.645748,0.115019,0.770701,0.360119,968,7448,288,5290
0.121153,0.423901,0.646172,0.117376,0.749204,0.360785,941,7076,315,5662
0.316228,0.397322,0.646102,0.111087,0.816083,0.359624,1025,8202,231,4536
0.825404,0.396073,0.645692,0.110716,0.818471,0.359214,1028,8257,228,4481
2.154435,0.394628,0.645298,0.110197,0.820860,0.358484,1031,8325,225,4413


## 4. Обучение финальной модели и выбор порога

Беру лучший `C` из шага 3, обучаю одну финальную модель и получаю `predict_proba` сразу для train, val и test.

Порог отсечения снова подбираю по F2 на val - выходит 0.4239. Это число фиксируется и потом применяется ко всем трём сплитам как есть.

Подбирать порог на test я специально не стал. Это была бы утечка: модель тогда фактически "посмотрела" бы в test через выбор порога, и итоговая метрика оказалась бы оптимистично завышенной. Корректный сценарий - подобрать на val, заморозить, применить к test как к новым данным.

И почему порог не классические 0.5? Из-за `class_weight="balanced"` шкала вероятностей сильно сдвинута: модель легко выдаёт значения выше 0.5 и для отрицательного класса. Поэтому оптимум по F2 уезжает в район 0.42.

In [5]:
final_pipe = Pipeline([
    ("prep", preprocessor),
    ("clf",  LogisticRegression(
        C=best_C, class_weight="balanced",
        max_iter=1000, random_state=SEED, solver="lbfgs"
    )),
])
final_pipe.fit(X_train, y_train)

proba_train = final_pipe.predict_proba(X_train)[:, 1]
proba_val   = final_pipe.predict_proba(X_val)[:, 1]
proba_test  = final_pipe.predict_proba(X_test)[:, 1]

# Порог выбирается по val и фиксируется для всех трёх сплитов
threshold = utils.find_best_threshold_f2(y_val, proba_val)
print(f"Финальный порог (по F2 на val): {threshold:.4f}")

Финальный порог (по F2 на val): 0.4239


## 5. Метрики


На test получилось так:

| Метрика   | Минимум | Хороший | Получили |   |
|-----------|--------:|--------:|---------:|:-:|
| ROC-AUC   |    0.64 |    0.67 |   0.6529 | ✓ |
| Precision |    0.15 |    0.20 |   0.1162 | ✗ |
| Recall    |    0.45 |    0.60 |   0.7514 | ✓ |
| F2        |    0.30 |    0.38 |   0.3590 | ✓ |

Если перевести в "сколько пациентов мы поймали": из всех, кто реально вернулся в течение 30 дней, модель пометила ~75%. Но из 100 пациентов, которых она пометила "риск", вернулись только 12, остальные 88 - ложные тревоги. На цифрах: tp=943, fp=7169, fn=312, tn=5570.

Низкий precision тут не баг, а следствие нашей же стратегии. Я намеренно перекосил модель в сторону recall: дал редкому классу повышенный вес и подобрал порог по F2. Логика простая - лучше зря всполошиться десять раз, чем один раз пропустить реально опасного пациента. Главная метрика у нас F2, и она в норме.

Переобучения нет: ROC-AUC на train 0.6657, на test 0.6529, разрыв 0.013 - это практически шум. L2-регуляризация делает своё дело.

В последней ячейке вероятности и метрики уходят на диск. Ноутбук 08 потом подхватит файлы и положит CatBoost, MLP, Transformer и TabM рядом для сводного сравнения.

In [6]:
m_train = utils.compute_metrics(y_train, proba_train, threshold)
m_val   = utils.compute_metrics(y_val,   proba_val,   threshold)
m_test  = utils.compute_metrics(y_test,  proba_test,  threshold)

metrics_df = pd.DataFrame(
    {"train": m_train, "val": m_val, "test": m_test}
).T.round(4)
print(metrics_df.to_string())

# Справочные целевые значения (для CatBoost/TabM);
# baseline-модель может быть ниже — это нормально.
targets = {"roc_auc": (0.64, 0.67), "precision": (0.15, 0.20),
           "recall": (0.45, 0.60), "f2": (0.30, 0.38)}
print("\n  метрика   min   хорошо  test")
for k, (mn, gd) in targets.items():
    v = m_test[k]
    flag = "✓" if v >= mn else "✗"
    print(f"  {k:<12}{mn:.2f}  {gd:.2f}   {v:.4f}  {flag}")

       roc_auc  precision  recall      f2      tp       fp     fn       tn
train   0.6657     0.1187  0.7647  0.3662  2880.0  21383.0  886.0  16833.0
val     0.6462     0.1174  0.7492  0.3608   941.0   7076.0  315.0   5662.0
test    0.6529     0.1162  0.7514  0.3590   943.0   7169.0  312.0   5570.0

  метрика   min   хорошо  test
  roc_auc     0.64  0.67   0.6529  ✓
  precision   0.15  0.20   0.1162  ✗
  recall      0.45  0.60   0.7514  ✓
  f2          0.30  0.38   0.3590  ✓


In [7]:
# Сохранение предсказаний и метрик
pd.DataFrame({"y_true": y_val,  "y_proba": proba_val}).to_csv(
    RESULTS_PREDS / "logreg_val.csv",  index=False
)
pd.DataFrame({"y_true": y_test, "y_proba": proba_test}).to_csv(
    RESULTS_PREDS / "logreg_test.csv", index=False
)

metrics_out = {
    "model": "logreg",
    "best_C": best_C,
    "threshold": threshold,
    "train": m_train,
    "val":   m_val,
    "test":  m_test,
}
with open(RESULTS_METRICS / "logreg.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

print("Артефакты сохранены:")
print("  results/predictions/logreg_val.csv")
print("  results/predictions/logreg_test.csv")
print("  results/metrics/logreg.json")

Артефакты сохранены:
  results/predictions/logreg_val.csv
  results/predictions/logreg_test.csv
  results/metrics/logreg.json
